# SchemaShift GRPO Training — Kaggle T4

**v2.1 configuration — three parallel experiments across team Kaggle accounts (TOS-compliant):**

- **Account 1 (Yashash — control):** Binary reward only. Tests whether sparse binary reward is sufficient.
- **Account 2 (Gajanand — main):** Full shaped reward (binary + rubric + dense step shaping).
- **Account 3 (Likith — curriculum):** Shaped reward + procedural drift scheduler (see `build_procedural_scenarios`).

Before running:
- Set `SCHEMASHIFT_URL` env var to your deployed HF Space (or ngrok-tunneled localhost for dev)
- Set `HF_TOKEN` in Kaggle secrets (for checkpoint push)
- Enable T4 x2 GPU in Kaggle session settings
- Enable Internet in notebook settings

In [ ]:
# Install deps
!pip install -q unsloth trl==0.13.0 vllm==0.6.3
!pip install -q httpx pydantic fastapi
!pip install -q openai  # action parser fallback

In [ ]:
# Clone repo + install editable
import os
os.chdir("/kaggle/working")
!git clone https://github.com/Yashash4/SchemaShift.git || (cd SchemaShift && git pull)
os.chdir("/kaggle/working/SchemaShift")
!pip install -e .

In [ ]:
# Load Qwen 2.5 1.5B + LoRA adapter
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=3072,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
print(f"Trainable params: {model.num_parameters(only_trainable=True):,}")

In [ ]:
# Env client + reward function (variant-specific — uncomment ONE block)
import os, json, re
from client import SchemaShiftEnvClient
from models import Action, CompleteParams

SCHEMASHIFT_URL = os.getenv("SCHEMASHIFT_URL", "https://YOUR-SPACE.hf.space")
env_client = SchemaShiftEnvClient(base_url=SCHEMASHIFT_URL)


def parse_completion_to_actions(text: str) -> list[Action]:
    """Parse LLM output into a list of Actions. Graceful fallback on any failure."""
    # Strip common markdown code fences
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", text)
    actions = []
    # Brace-balanced extractor (handles arbitrary nesting, unlike flat regex)
    depth = 0
    start = -1
    for i, ch in enumerate(cleaned):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start >= 0:
                chunk = cleaned[start:i + 1]
                try:
                    obj = json.loads(chunk)
                    actions.append(Action.model_validate(obj))
                except Exception:
                    pass
                start = -1
    if not actions:
        actions.append(Action(
            type="complete_task",
            complete=CompleteParams(summary="Parse error — no valid actions in output."),
        ))
    return actions


# --- ACCOUNT 1 (Yashash — control): binary only ---
# def reward_fn(prompts, completions, **kwargs):
#     rewards = []
#     for prompt, completion in zip(prompts, completions):
#         actions = parse_completion_to_actions(completion)
#         task_id = kwargs.get("task_id", "E1_onboard_new_hire")
#         env_client.reset(task_id)
#         last_reward = 0.0
#         for a in actions:
#             obs, r = env_client.step(a)
#             last_reward = r.binary
#             if obs.done:
#                 break
#         rewards.append(last_reward)
#     return rewards


# --- ACCOUNT 2 (Gajanand — main): shaped total ---
def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        actions = parse_completion_to_actions(completion)
        task_id = kwargs.get("task_id", "E1_onboard_new_hire")
        env_client.reset(task_id)
        total = 0.0
        for a in actions:
            obs, r = env_client.step(a)
            total += r.shaped_total
            if obs.done:
                break
        rewards.append(total)
    return rewards


# --- ACCOUNT 3 (Likith — curriculum with procedural scheduler) ---
# See next cell for build_procedural_scenarios. Rebuild dataset every ~25 steps.


In [ ]:
# Procedural drift scheduler (Account 3 only — comment out for Accounts 1 & 2)
import random
from copy import deepcopy
from scenarios import SCENARIOS
from models import DriftEvent


def build_procedural_scenarios(step_count: int) -> dict:
    """Curriculum-based drift variety that scales with training progress.

    - Steps 0-50:   Original deterministic drifts (baseline)
    - Steps 50-150: Permute drift steps (same drifts, shuffled timing)
    - Steps 150+:   Add a secondary drift (harder, multi-drift episodes)
    """
    scenarios_out = {}
    for task_id, scenario in SCENARIOS.items():
        if scenario["difficulty"] != "easy":
            scenarios_out[task_id] = scenario
            continue

        s = deepcopy(scenario)

        if step_count < 50:
            pass
        elif step_count < 150:
            max_step = s["max_steps"]
            for d in s["drift_plan"]:
                d.fires_at_step = random.randint(1, max(1, max_step - 2))
                d.details["_fired"] = False
        else:
            if s["drift_plan"]:
                primary = s["drift_plan"][0]
                secondary_kinds = ["rate_limit_tightening", "error_code_remap"]
                secondary = DriftEvent(
                    tool=primary.tool,
                    endpoint=None,
                    kind=random.choice(secondary_kinds),
                    fires_at_step=min(s["max_steps"] - 1, primary.fires_at_step + 2),
                    details={"_procedural": True},
                )
                s["drift_plan"].append(secondary)

        scenarios_out[task_id] = s
    return scenarios_out

In [ ]:
# Prompt dataset builder
from scenarios import SCENARIOS


def build_prompts_dataset(scenarios_dict):
    prompts = []
    for task_id, scenario in scenarios_dict.items():
        prompt = f"""You are an autonomous workflow agent. Complete the task using tools.

TASK: {scenario['task_description']}

SUCCESS CRITERIA:
{chr(10).join(f'- {c}' for c in scenario['success_criteria'])}

AVAILABLE ACTIONS:
- call_tool(tool, endpoint, params) — call a tool
- inspect_schema(tool) — read current schema for a tool
- retry_with_variant(tool, endpoint, params) — retry a failed call with modified params
- report_drift(tool, drift_kind, description) — flag a drift you detected
- complete_task(summary) — end the episode

Output your next action as JSON. Example:
{{"type": "call_tool", "tool_call": {{"tool": "mail", "endpoint": "send_message", "params": {{"to": "x@y.com", "subject": "Hi", "body": "Hello"}}}}}}
"""
        prompts.append({"prompt": prompt, "task_id": task_id})
    return prompts


# Accounts 1 & 2 (deterministic scenarios):
train_dataset = build_prompts_dataset(SCENARIOS)

# Account 3 (procedural — rebuild each training chunk):
# train_dataset = build_prompts_dataset(build_procedural_scenarios(step_count=0))

In [ ]:
# GRPO config (Kaggle T4-friendly)
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

config = GRPOConfig(
    output_dir="schemashift-grpo-kaggle",
    num_generations=4,
    max_completion_length=1536,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=5,
    save_steps=25,
    max_steps=100,
    report_to="none",
    use_vllm=True,
    vllm_mode="colocate",
    push_to_hub=True,
    hub_model_id=f"{os.getenv('HF_USERNAME', 'Yashash4')}/schemashift-qwen15b-kaggle",
)

dataset = Dataset.from_list(train_dataset)

In [ ]:
# Train
trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=[reward_fn],
    train_dataset=dataset,
    processing_class=tokenizer,
)

# ABORT-EARLY CHECK: if first 5 logged rewards are all 0, STOP — reward is broken.
trainer.train()

In [ ]:
# Save + push final checkpoint
# trainer.save_model("schemashift-qwen15b-final")
# trainer.push_to_hub()
print("Training complete. Checkpoint saved locally; HF push configured.")